In [2]:
DATA_DIR = r"D:\Haseeb\Datasets\pacs_data"

In [2]:
import os
import csv
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from dataset import get_pacs_dataloaders
from utils import *
from pruning import iterative_pruning

ALL_DOMAINS = ['art_painting', 'cartoon', 'photo', 'sketch']
BATCH_SIZE = 256
NUM_WORKERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PRUNE_RATES = [0.10, 0.10, 0.10]
FINETUNE_EPOCHS = 5
FINETUNE_LR = 1e-4
ALPHA = 1.0


# CSV output
CSV_PATH = "pacs_pruning_results.csv"
CSV_FIELDS = [
    "target_domain",
    "warmup_model",
    "pruned_model",
    "mask_file",
    "best_warmup_acc",
    "final_pruned_acc",
    "improvement",
    "percent_pruned"   # percent of parameters removed (sparsity)
]


def ensure_csv_header(path, fields):
    if not os.path.exists(path):
        with open(path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fields)
            writer.writeheader()


def append_result_to_csv(path, row, fields):
    with open(path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writerow(row)


def compute_percent_pruned(mask):
    total = 0
    zeros = 0
    for k, v in mask.items():
        if not isinstance(v, torch.Tensor):
            try:
                v = torch.tensor(v)
            except Exception:
                continue
        total += v.numel()
        zeros += int((v == 0).sum().item())
    if total == 0:
        return 0.0
    percent_pruned = 100.0 * zeros / total
    return percent_pruned


def run_for_target_domain(TARGET_DOMAIN):
    print("\n" + "=" * 70)
    print(f"RUNNING PIPELINE FOR TARGET DOMAIN: {TARGET_DOMAIN}")
    print("=" * 70 + "\n")

    SOURCE_DOMAINS = [d for d in ALL_DOMAINS if d != TARGET_DOMAIN]
    WARMUP_MODEL_PATH = f"warmup_{TARGET_DOMAIN}.pth"
    PRUNED_MODEL_PATH = f"pruned_{TARGET_DOMAIN}.pth"
    PRUNED_MASK_PATH = f"mask_{TARGET_DOMAIN}.pth"

    # ---------- DATALOADERS ----------
    source_loader_combined, target_loader, class_to_idx = get_pacs_dataloaders(
        data_dir=DATA_DIR,
        source_domains=SOURCE_DOMAINS,
        target_domain=TARGET_DOMAIN,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        combine_sources=True
    )
    num_classes = len(class_to_idx)

    # ---------- WARMUP ----------
    best_warmup_acc = 0.0
    if os.path.exists(WARMUP_MODEL_PATH):
        print(f"Found existing warmup checkpoint: {WARMUP_MODEL_PATH}. Loading.")
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        model.load_state_dict(torch.load(WARMUP_MODEL_PATH, map_location=DEVICE))
        model.to(DEVICE)
        # optional: evaluate loaded model to populate best_warmup_acc
        _, best_warmup_acc = evaluate(model, target_loader, DEVICE)
        print(f"  Warmup loaded eval -> {best_warmup_acc:.2f}%")
    else:
        print("No warmup checkpoint found. Running warmup training.")
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        model.to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=1e-4)
        WARMUP_EPOCHS = 5
        for epoch in range(WARMUP_EPOCHS):
            train_vanilla(model, source_loader_combined, optimizer, DEVICE, epoch)
            _, val_acc = evaluate(model, target_loader, DEVICE)
            print(f"  Warmup Epoch {epoch+1}: Target Accuracy = {val_acc:.2f}%")
            if val_acc > best_warmup_acc:
                best_warmup_acc = val_acc
                torch.save(model.state_dict(), WARMUP_MODEL_PATH)
                print(f"    Saved new best warmup ({best_warmup_acc:.2f}%).")

    # ---------- PRUNING ----------
    print("\nStarting iterative pruning...")
    source_loaders_list, target_loader, _ = get_pacs_dataloaders(
        data_dir=DATA_DIR,
        source_domains=SOURCE_DOMAINS,
        target_domain=TARGET_DOMAIN,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        combine_sources=False
    )

    pruning_model = models.resnet18()
    pruning_model.fc = nn.Linear(pruning_model.fc.in_features, num_classes)
    pruning_model.load_state_dict(torch.load(WARMUP_MODEL_PATH, map_location=DEVICE))
    pruning_model.to(DEVICE)

    final_model, final_mask = iterative_pruning(
        model=pruning_model,
        source_loaders_list=source_loaders_list,
        target_loader=target_loader,
        device=DEVICE,
        prune_rates=PRUNE_RATES,
        retrain_epochs=FINETUNE_EPOCHS,
        lr=FINETUNE_LR,
        alpha=ALPHA,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        SFT=True,
        importance_type="taylor",
        keep_overall_best=False
    )

    # ---------- FINAL EVAL ----------
    apply_mask(final_model, final_mask)
    _, final_acc = evaluate(final_model, target_loader, DEVICE, mask=final_mask)
    percent_pruned = compute_percent_pruned(final_mask)

    print("\n--- Summary for", TARGET_DOMAIN, "---")
    print(f"Warmup best acc: {best_warmup_acc:.2f}%")
    print(f"Final pruned acc: {final_acc:.2f}%")
    print(f"Improvement: {final_acc - best_warmup_acc:+.2f}%")
    print(f"Percent pruned (zeros in mask): {percent_pruned:.2f}%")

    # Save artifacts
    torch.save(final_model.state_dict(), PRUNED_MODEL_PATH)
    torch.save(final_mask, PRUNED_MASK_PATH)

    # Append result row to CSV
    row = {
        "target_domain": TARGET_DOMAIN,
        "warmup_model": os.path.abspath(WARMUP_MODEL_PATH),
        "pruned_model": os.path.abspath(PRUNED_MODEL_PATH),
        "mask_file": os.path.abspath(PRUNED_MASK_PATH),
        "best_warmup_acc": f"{best_warmup_acc:.4f}",
        "final_pruned_acc": f"{final_acc:.4f}",
        "improvement": f"{(final_acc - best_warmup_acc):.4f}",
        "percent_pruned": f"{percent_pruned:.4f}"
    }
    append_result_to_csv(CSV_PATH, row, CSV_FIELDS)
    print(f"Results appended to {CSV_PATH}")

    return row


ensure_csv_header(CSV_PATH, CSV_FIELDS)
all_results = []
for domain in ALL_DOMAINS:
    try:
        res = run_for_target_domain(domain)
        all_results.append(res)
    except Exception as e:
        print(f"ERROR while processing {domain}: {e}")
        continue

print("\nALL RUNS COMPLETE. Summary rows:")
for r in all_results:
    print(r)
print(f"\nCSV saved at: {os.path.abspath(CSV_PATH)}")



RUNNING PIPELINE FOR TARGET DOMAIN: art_painting

Creating source datasets for: ['cartoon', 'photo', 'sketch']
  - Domain 'cartoon' (ID 0) loaded with 2344 images.
  - Domain 'photo' (ID 1) loaded with 1670 images.
  - Domain 'sketch' (ID 2) loaded with 3929 images.
Combined source dataloader created with 7943 total images.
Creating target dataloader for: art_painting
  - Domain 'art_painting' loaded with 2048 images.
No warmup checkpoint found. Running warmup training.


Epoch 1 Vanilla Training:   0%|          | 0/31 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

  Warmup Epoch 1: Target Accuracy = 70.31%
    Saved new best warmup (70.31%).


Epoch 2 Vanilla Training:   0%|          | 0/31 [00:00<?, ?it/s]

...

Epoch 5 Train2 (Normal):   0%|          | 0/24 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

  Epoch 5 Target Accuracy: 62.43%
Iteration 3 | Best Accuracy in this round: 63.12%


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]


--- Summary for sketch ---
Warmup best acc: 63.30%
Final pruned acc: 63.12%
Improvement: -0.18%
Percent pruned (zeros in mask): 25.93%
Results appended to pacs_pruning_results.csv

ALL RUNS COMPLETE. Summary rows:
{'target_domain': 'art_painting', 'warmup_model': 'd:\\Haseeb\\pruning_sproj\\warmup_art_painting.pth', 'pruned_model': 'd:\\Haseeb\\pruning_sproj\\pruned_art_painting.pth', 'mask_file': 'd:\\Haseeb\\pruning_sproj\\mask_art_painting.pth', 'best_warmup_acc': '75.7324', 'final_pruned_acc': '73.0469', 'improvement': '-2.6855', 'percent_pruned': '25.9251'}
{'target_domain': 'cartoon', 'warmup_model': 'd:\\Haseeb\\pruning_sproj\\warmup_cartoon.pth', 'pruned_model': 'd:\\Haseeb\\pruning_sproj\\pruned_cartoon.pth', 'mask_file': 'd:\\Haseeb\\pruning_sproj\\mask_cartoon.pth', 'best_warmup_acc': '71.2031', 'final_pruned_acc': '72.0563', 'improvement': '0.8532', 'percent_pruned': '25.9251'}
{'target_domain': 'photo', 'warmup_model': 'd:\\Haseeb\\pruning_sproj\\warmup_photo.pth', 'prune

In [3]:
import os
import csv
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from dataset import get_pacs_dataloaders
from utils import *
from pruning import iterative_pruning

ALL_DOMAINS = ['art_painting', 'cartoon', 'photo', 'sketch']
BATCH_SIZE = 256
NUM_WORKERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PRUNE_RATES = [0.10, 0.10, 0.10]
FINETUNE_EPOCHS = 5
FINETUNE_LR = 1e-4
ALPHA = 1.0


# CSV output
CSV_PATH = "pacs_pruning_results.csv"
CSV_FIELDS = [
    "target_domain",
    "warmup_model",
    "pruned_model",
    "mask_file",
    "best_warmup_acc",
    "final_pruned_acc",
    "improvement",
    "percent_pruned"   # percent of parameters removed (sparsity)
]


def ensure_csv_header(path, fields):
    if not os.path.exists(path):
        with open(path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fields)
            writer.writeheader()


def append_result_to_csv(path, row, fields):
    with open(path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writerow(row)


def compute_percent_pruned(mask):
    total = 0
    zeros = 0
    for k, v in mask.items():
        if not isinstance(v, torch.Tensor):
            try:
                v = torch.tensor(v)
            except Exception:
                continue
        total += v.numel()
        zeros += int((v == 0).sum().item())
    if total == 0:
        return 0.0
    percent_pruned = 100.0 * zeros / total
    return percent_pruned


def run_for_target_domain(TARGET_DOMAIN):
    print("\n" + "=" * 70)
    print(f"RUNNING PIPELINE FOR TARGET DOMAIN: {TARGET_DOMAIN}")
    print("=" * 70 + "\n")

    SOURCE_DOMAINS = [d for d in ALL_DOMAINS if d != TARGET_DOMAIN]
    WARMUP_MODEL_PATH = f"warmup_{TARGET_DOMAIN}.pth"
    PRUNED_MODEL_PATH = f"pruned_{TARGET_DOMAIN}.pth"
    PRUNED_MASK_PATH = f"mask_{TARGET_DOMAIN}.pth"

    # ---------- DATALOADERS ----------
    source_loader_combined, target_loader, class_to_idx = get_pacs_dataloaders(
        data_dir=DATA_DIR,
        source_domains=SOURCE_DOMAINS,
        target_domain=TARGET_DOMAIN,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        combine_sources=True
    )
    num_classes = len(class_to_idx)

    # ---------- WARMUP ----------
    best_warmup_acc = 0.0
    if os.path.exists(WARMUP_MODEL_PATH):
        print(f"Found existing warmup checkpoint: {WARMUP_MODEL_PATH}. Loading.")
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        model.load_state_dict(torch.load(WARMUP_MODEL_PATH, map_location=DEVICE))
        model.to(DEVICE)
        # optional: evaluate loaded model to populate best_warmup_acc
        _, best_warmup_acc = evaluate(model, target_loader, DEVICE)
        print(f"  Warmup loaded eval -> {best_warmup_acc:.2f}%")
    else:
        print("No warmup checkpoint found. Running warmup training.")
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        model.to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=1e-4)
        WARMUP_EPOCHS = 5
        for epoch in range(WARMUP_EPOCHS):
            train_vanilla(model, source_loader_combined, optimizer, DEVICE, epoch)
            _, val_acc = evaluate(model, target_loader, DEVICE)
            print(f"  Warmup Epoch {epoch+1}: Target Accuracy = {val_acc:.2f}%")
            if val_acc > best_warmup_acc:
                best_warmup_acc = val_acc
                torch.save(model.state_dict(), WARMUP_MODEL_PATH)
                print(f"    Saved new best warmup ({best_warmup_acc:.2f}%).")

    # ---------- PRUNING ----------
    print("\nStarting iterative pruning...")
    source_loaders_list, target_loader, _ = get_pacs_dataloaders(
        data_dir=DATA_DIR,
        source_domains=SOURCE_DOMAINS,
        target_domain=TARGET_DOMAIN,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        combine_sources=False
    )

    pruning_model = models.resnet18()
    pruning_model.fc = nn.Linear(pruning_model.fc.in_features, num_classes)
    pruning_model.load_state_dict(torch.load(WARMUP_MODEL_PATH, map_location=DEVICE))
    pruning_model.to(DEVICE)

    final_model, final_mask = iterative_pruning(
        model=pruning_model,
        source_loaders_list=source_loaders_list,
        target_loader=target_loader,
        device=DEVICE,
        prune_rates=PRUNE_RATES,
        retrain_epochs=FINETUNE_EPOCHS,
        lr=FINETUNE_LR,
        alpha=ALPHA,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        SFT=True,
        importance_type="taylor",
        keep_overall_best=True
    )

    # ---------- FINAL EVAL ----------
    apply_mask(final_model, final_mask)
    _, final_acc = evaluate(final_model, target_loader, DEVICE, mask=final_mask)
    percent_pruned = compute_percent_pruned(final_mask)

    print("\n--- Summary for", TARGET_DOMAIN, "---")
    print(f"Warmup best acc: {best_warmup_acc:.2f}%")
    print(f"Final pruned acc: {final_acc:.2f}%")
    print(f"Improvement: {final_acc - best_warmup_acc:+.2f}%")
    print(f"Percent pruned (zeros in mask): {percent_pruned:.2f}%")

    # Save artifacts
    torch.save(final_model.state_dict(), PRUNED_MODEL_PATH)
    torch.save(final_mask, PRUNED_MASK_PATH)

    # Append result row to CSV
    row = {
        "target_domain": TARGET_DOMAIN,
        "warmup_model": os.path.abspath(WARMUP_MODEL_PATH),
        "pruned_model": os.path.abspath(PRUNED_MODEL_PATH),
        "mask_file": os.path.abspath(PRUNED_MASK_PATH),
        "best_warmup_acc": f"{best_warmup_acc:.4f}",
        "final_pruned_acc": f"{final_acc:.4f}",
        "improvement": f"{(final_acc - best_warmup_acc):.4f}",
        "percent_pruned": f"{percent_pruned:.4f}"
    }
    append_result_to_csv(CSV_PATH, row, CSV_FIELDS)
    print(f"Results appended to {CSV_PATH}")

    return row


ensure_csv_header(CSV_PATH, CSV_FIELDS)
all_results = []
for domain in ALL_DOMAINS:
    try:
        res = run_for_target_domain(domain)
        all_results.append(res)
    except Exception as e:
        print(f"ERROR while processing {domain}: {e}")
        continue

print("\nALL RUNS COMPLETE. Summary rows:")
for r in all_results:
    print(r)
print(f"\nCSV saved at: {os.path.abspath(CSV_PATH)}")



RUNNING PIPELINE FOR TARGET DOMAIN: art_painting

Creating source datasets for: ['cartoon', 'photo', 'sketch']
  - Domain 'cartoon' (ID 0) loaded with 2344 images.
  - Domain 'photo' (ID 1) loaded with 1670 images.
  - Domain 'sketch' (ID 2) loaded with 3929 images.
Combined source dataloader created with 7943 total images.
Creating target dataloader for: art_painting
  - Domain 'art_painting' loaded with 2048 images.
Found existing warmup checkpoint: warmup_art_painting.pth. Loading.


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

  Warmup loaded eval -> 75.73%

Starting iterative pruning...
Creating source datasets for: ['cartoon', 'photo', 'sketch']
  - Domain 'cartoon' (ID 0) loaded with 2344 images.
  - Domain 'photo' (ID 1) loaded with 1670 images.
  - Domain 'sketch' (ID 2) loaded with 3929 images.
Created 3 separate source dataloaders.
Creating target dataloader for: art_painting
  - Domain 'art_painting' loaded with 2048 images.


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

Initial Baseline Target Accuracy: 75.73%

--- Pruning Iteration 1/3 with base rate 0.1 ---

Generating mask with base iterative prune rate: 0.1
  - Layer 'conv1': Pruning 1/64 active filters (rate 0.025).
  - Layer 'layer1.0.conv1': Pruning 1/64 active filters (rate 0.025).
  - Layer 'layer1.0.conv2': Pruning 1/64 active filters (rate 0.025).
  - Layer 'layer1.1.conv1': Pruning 1/64 active filters (rate 0.025).
  - Layer 'layer1.1.conv2': Pruning 1/64 active filters (rate 0.025).
  - Layer 'layer2.0.conv1': Pruning 6/128 active filters (rate 0.050).
  - Layer 'layer2.0.conv2': Pruning 6/128 active filters (rate 0.050).
  - Layer 'layer2.0.downsample.0': Pruning 6/128 active filters (rate 0.050).
  - Layer 'layer2.1.conv1': Pruning 6/128 active filters (rate 0.050).
  - Layer 'layer2.1.conv2': Pruning 6/128 active filters (rate 0.050).
  - Layer 'layer3.0.conv1': Pruning 25/256 active filters (rate 0.100).
  - Layer 'layer3.0.conv2': Pruning 25/256 active filters (rate 0.100).
  - Layer

...

Epoch 5 Train2 (Normal):   0%|          | 0/24 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

  Epoch 5 Target Accuracy: 57.50%
Iteration 3 | Best Accuracy in this round: 57.93%
Overall Best Accuracy so far: 63.78%


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]


--- Summary for sketch ---
Warmup best acc: 63.30%
Final pruned acc: 19.17%
Improvement: -44.13%
Percent pruned (zeros in mask): 25.93%
Results appended to pacs_pruning_results.csv

ALL RUNS COMPLETE. Summary rows:
{'target_domain': 'art_painting', 'warmup_model': 'd:\\Haseeb\\pruning_sproj\\warmup_art_painting.pth', 'pruned_model': 'd:\\Haseeb\\pruning_sproj\\pruned_art_painting.pth', 'mask_file': 'd:\\Haseeb\\pruning_sproj\\mask_art_painting.pth', 'best_warmup_acc': '75.7324', 'final_pruned_acc': '14.7949', 'improvement': '-60.9375', 'percent_pruned': '25.9251'}
{'target_domain': 'cartoon', 'warmup_model': 'd:\\Haseeb\\pruning_sproj\\warmup_cartoon.pth', 'pruned_model': 'd:\\Haseeb\\pruning_sproj\\pruned_cartoon.pth', 'mask_file': 'd:\\Haseeb\\pruning_sproj\\mask_cartoon.pth', 'best_warmup_acc': '71.2031', 'final_pruned_acc': '50.2560', 'improvement': '-20.9471', 'percent_pruned': '25.9251'}
{'target_domain': 'photo', 'warmup_model': 'd:\\Haseeb\\pruning_sproj\\warmup_photo.pth', 'p

In [3]:
import os
import csv
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from dataset import get_pacs_dataloaders
from utils import *
from pruning import iterative_pruning

ALL_DOMAINS = ['art_painting', 'cartoon', 'photo', 'sketch']
BATCH_SIZE = 256
NUM_WORKERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PRUNE_RATES = [0.10, 0.10, 0.10]
FINETUNE_EPOCHS = 5
FINETUNE_LR = 1e-4
ALPHA = 1.0


# CSV output
CSV_PATH = "pacs_pruning_results.csv"
CSV_FIELDS = [
    "target_domain",
    "warmup_model",
    "pruned_model",
    "mask_file",
    "best_warmup_acc",
    "final_pruned_acc",
    "improvement",
    "percent_pruned"   # percent of parameters removed (sparsity)
]


def ensure_csv_header(path, fields):
    if not os.path.exists(path):
        with open(path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fields)
            writer.writeheader()


def append_result_to_csv(path, row, fields):
    with open(path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writerow(row)


def compute_percent_pruned(mask):
    total = 0
    zeros = 0
    for k, v in mask.items():
        if not isinstance(v, torch.Tensor):
            try:
                v = torch.tensor(v)
            except Exception:
                continue
        total += v.numel()
        zeros += int((v == 0).sum().item())
    if total == 0:
        return 0.0
    percent_pruned = 100.0 * zeros / total
    return percent_pruned


def run_for_target_domain(TARGET_DOMAIN):
    print("\n" + "=" * 70)
    print(f"RUNNING PIPELINE FOR TARGET DOMAIN: {TARGET_DOMAIN}")
    print("=" * 70 + "\n")

    SOURCE_DOMAINS = [d for d in ALL_DOMAINS if d != TARGET_DOMAIN]
    WARMUP_MODEL_PATH = f"warmup_{TARGET_DOMAIN}.pth"
    PRUNED_MODEL_PATH = f"pruned_{TARGET_DOMAIN}.pth"
    PRUNED_MASK_PATH = f"mask_{TARGET_DOMAIN}.pth"

    # ---------- DATALOADERS ----------
    source_loader_combined, target_loader, class_to_idx = get_pacs_dataloaders(
        data_dir=DATA_DIR,
        source_domains=SOURCE_DOMAINS,
        target_domain=TARGET_DOMAIN,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        combine_sources=True
    )
    num_classes = len(class_to_idx)

    # ---------- WARMUP ----------
    best_warmup_acc = 0.0
    if os.path.exists(WARMUP_MODEL_PATH):
        print(f"Found existing warmup checkpoint: {WARMUP_MODEL_PATH}. Loading.")
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        model.load_state_dict(torch.load(WARMUP_MODEL_PATH, map_location=DEVICE))
        model.to(DEVICE)
        # optional: evaluate loaded model to populate best_warmup_acc
        _, best_warmup_acc = evaluate(model, target_loader, DEVICE)
        print(f"  Warmup loaded eval -> {best_warmup_acc:.2f}%")
    else:
        print("No warmup checkpoint found. Running warmup training.")
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        model.to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=1e-4)
        WARMUP_EPOCHS = 5
        for epoch in range(WARMUP_EPOCHS):
            train_vanilla(model, source_loader_combined, optimizer, DEVICE, epoch)
            _, val_acc = evaluate(model, target_loader, DEVICE)
            print(f"  Warmup Epoch {epoch+1}: Target Accuracy = {val_acc:.2f}%")
            if val_acc > best_warmup_acc:
                best_warmup_acc = val_acc
                torch.save(model.state_dict(), WARMUP_MODEL_PATH)
                print(f"    Saved new best warmup ({best_warmup_acc:.2f}%).")

    # ---------- PRUNING ----------
    print("\nStarting iterative pruning...")
    source_loaders_list, target_loader, _ = get_pacs_dataloaders(
        data_dir=DATA_DIR,
        source_domains=SOURCE_DOMAINS,
        target_domain=TARGET_DOMAIN,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        combine_sources=False
    )

    pruning_model = models.resnet18()
    pruning_model.fc = nn.Linear(pruning_model.fc.in_features, num_classes)
    pruning_model.load_state_dict(torch.load(WARMUP_MODEL_PATH, map_location=DEVICE))
    pruning_model.to(DEVICE)

    final_model, final_mask = iterative_pruning(
        model=pruning_model,
        source_loaders_list=source_loaders_list,
        target_loader=target_loader,
        device=DEVICE,
        prune_rates=PRUNE_RATES,
        retrain_epochs=FINETUNE_EPOCHS,
        lr=FINETUNE_LR,
        alpha=ALPHA,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        SFT=False,
        importance_type="taylor",
        keep_overall_best=True
    )

    # ---------- FINAL EVAL ----------
    apply_mask(final_model, final_mask)
    _, final_acc = evaluate(final_model, target_loader, DEVICE, mask=final_mask)
    percent_pruned = compute_percent_pruned(final_mask)

    print("\n--- Summary for", TARGET_DOMAIN, "---")
    print(f"Warmup best acc: {best_warmup_acc:.2f}%")
    print(f"Final pruned acc: {final_acc:.2f}%")
    print(f"Improvement: {final_acc - best_warmup_acc:+.2f}%")
    print(f"Percent pruned (zeros in mask): {percent_pruned:.2f}%")

    # Save artifacts
    torch.save(final_model.state_dict(), PRUNED_MODEL_PATH)
    torch.save(final_mask, PRUNED_MASK_PATH)

    # Append result row to CSV
    row = {
        "target_domain": TARGET_DOMAIN,
        "warmup_model": os.path.abspath(WARMUP_MODEL_PATH),
        "pruned_model": os.path.abspath(PRUNED_MODEL_PATH),
        "mask_file": os.path.abspath(PRUNED_MASK_PATH),
        "best_warmup_acc": f"{best_warmup_acc:.4f}",
        "final_pruned_acc": f"{final_acc:.4f}",
        "improvement": f"{(final_acc - best_warmup_acc):.4f}",
        "percent_pruned": f"{percent_pruned:.4f}"
    }
    append_result_to_csv(CSV_PATH, row, CSV_FIELDS)
    print(f"Results appended to {CSV_PATH}")

    return row


ensure_csv_header(CSV_PATH, CSV_FIELDS)
all_results = []
for domain in ALL_DOMAINS:
    try:
        res = run_for_target_domain(domain)
        all_results.append(res)
    except Exception as e:
        print(f"ERROR while processing {domain}: {e}")
        continue

print("\nALL RUNS COMPLETE. Summary rows:")
for r in all_results:
    print(r)
print(f"\nCSV saved at: {os.path.abspath(CSV_PATH)}")



RUNNING PIPELINE FOR TARGET DOMAIN: art_painting

Creating source datasets for: ['cartoon', 'photo', 'sketch']
  - Domain 'cartoon' (ID 0) loaded with 2344 images.
  - Domain 'photo' (ID 1) loaded with 1670 images.
  - Domain 'sketch' (ID 2) loaded with 3929 images.
Combined source dataloader created with 7943 total images.
Creating target dataloader for: art_painting
  - Domain 'art_painting' loaded with 2048 images.
No warmup checkpoint found. Running warmup training.


Epoch 1 Vanilla Training:   0%|          | 0/31 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

  Warmup Epoch 1: Target Accuracy = 71.00%
    Saved new best warmup (71.00%).


Epoch 2 Vanilla Training:   0%|          | 0/31 [00:00<?, ?it/s]

...

Epoch 5 Training:   0%|          | 0/24 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

  Epoch 5 Target Accuracy: 63.65%
Iteration 3 | Best Accuracy in this round: 63.65%
Overall Best Accuracy so far: 71.98%


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]


--- Summary for sketch ---
Warmup best acc: 68.34%
Final pruned acc: 13.16%
Improvement: -55.18%
Percent pruned (zeros in mask): 25.93%
Results appended to pacs_pruning_results.csv

ALL RUNS COMPLETE. Summary rows:
{'target_domain': 'art_painting', 'warmup_model': 'd:\\Haseeb\\pruning_sproj\\warmup_art_painting.pth', 'pruned_model': 'd:\\Haseeb\\pruning_sproj\\pruned_art_painting.pth', 'mask_file': 'd:\\Haseeb\\pruning_sproj\\mask_art_painting.pth', 'best_warmup_acc': '74.3652', 'final_pruned_acc': '18.3594', 'improvement': '-56.0059', 'percent_pruned': '25.9251'}
{'target_domain': 'cartoon', 'warmup_model': 'd:\\Haseeb\\pruning_sproj\\warmup_cartoon.pth', 'pruned_model': 'd:\\Haseeb\\pruning_sproj\\pruned_cartoon.pth', 'mask_file': 'd:\\Haseeb\\pruning_sproj\\mask_cartoon.pth', 'best_warmup_acc': '70.3925', 'final_pruned_acc': '56.1007', 'improvement': '-14.2918', 'percent_pruned': '25.9251'}
{'target_domain': 'photo', 'warmup_model': 'd:\\Haseeb\\pruning_sproj\\warmup_photo.pth', 'p